# WordCount

The canonical streaming example. Read lines of text, split them into words, and maintain a live word to count histogram. This is the Kafka Streams WordCount topology written as a DBSP circuit.

The Kafka Streams version splits each line into words with flatMapValues, regroups by word, and counts. Each stage has a direct counterpart here. The line stream becomes an Input carrying a Z-set, where each push is one batch of lines. The flatMapValues split becomes a linear Lift1 from a Z-set of lines to a Z-set of words. The groupBy and count become an Integrate, because a Z-set weight already is a count. The resulting KTable is the integrated word Z-set, mapping each word to its count.

Two ideas anchor the translation. Each push is one commit interval. It folds however many lines you hand it into a single update, the same role that commit.interval.ms plays in Kafka Streams. And a KTable is an Integrate, so its changelog is the per-tick delta, because differentiating an integral returns the original stream.

## The flatMap stage

flatMapValues is the only value logic in the pipeline, and it is linear. A line with weight w contributes its words with weight w, so the map distributes over Z-set addition. A linear map is already incremental as a single Lift1, with no Integrate or Differentiate around it.

In [1]:
import re

from pydbsp.circuit import Circuit
from pydbsp.compute import ComputeCtx
from pydbsp.core import Antichain, dbsp_time
from pydbsp.evaluate import Evaluator
from pydbsp.operator import Input, Integrate, Lift1, LiftStreamIntroduction
from pydbsp.progress import Time
from pydbsp.storage import DictStorage
from pydbsp.zset import ZSet, ZSetAddition

WORD = re.compile(r"\W+", re.UNICODE)


def to_words(lines: ZSet[str]) -> ZSet[str]:
    """flatMapValues: a linear Z-set map from a bag of lines to a bag of
    words. Each line's weight scales the words it emits."""
    out: dict[str, int] = {}
    for line, w in lines.items():
        for tok in WORD.split(line.lower()):
            if tok:
                out[tok] = out.get(tok, 0) + w
    return ZSet({k: v for k, v in out.items() if v != 0})


LINES = [
    "hello kafka streams",
    "all streams lead to kafka",
    "join kafka summit",
]

## Counting is integrating

A Z-set weight already is a count, so the group-by and the count collapse into a single Integrate. The cumulative word Z-set is the histogram. The circuit is three nodes and costs one pass over each batch of words.

In [2]:
g = ZSetAddition[str]()
e = Evaluator[Time](
    circuit=Circuit[Time](),
    storage=DictStorage(),
    ctx=ComputeCtx(lattice=dbsp_time(1)),
    group=g,
)

lines = Input[ZSet[str]](frontier=Antichain(dbsp_time(1))).connect(e.circuit, ())
words = Lift1[ZSet[str], ZSet[str]](f=to_words).connect(e.circuit, (lines,))
counts = Integrate[ZSet[str]](group=g).connect(e.circuit, (words,))

for line in LINES:
    e.push(lines, ZSet({line: 1}))

dict(sorted(e.latest(counts).inner.items()))

{'all': 1,
 'hello': 1,
 'join': 1,
 'kafka': 3,
 'lead': 1,
 'streams': 2,
 'summit': 1,
 'to': 1}

That is the Kafka Streams output histogram. kafka is 3, streams is 2, and the rest are 1.

## The changelog

Kafka Streams emits one update per changed word per commit. Pushing one line at a time reproduces this. The per-tick words delta names the words that changed, and the integrated counts give their new values. Reading them together yields the familiar kafka 1, kafka 2, kafka 3 progression.

In [3]:
g = ZSetAddition[str]()
e = Evaluator[Time](
    circuit=Circuit[Time](),
    storage=DictStorage(),
    ctx=ComputeCtx(lattice=dbsp_time(1)),
    group=g,
)
lines = Input[ZSet[str]](frontier=Antichain(dbsp_time(1))).connect(e.circuit, ())
words = Lift1[ZSet[str], ZSet[str]](f=to_words).connect(e.circuit, (lines,))
counts = Integrate[ZSet[str]](group=g).connect(e.circuit, (words,))

for t, line in enumerate(LINES):
    e.push(lines, ZSet({line: 1}))
    changed = e.read(words, (t,))
    live = e.read(counts, (t,))
    print(f"tick {t}: {line!r}")
    for word in changed.inner:
        print(f"    {word:>8} {live.inner[word]}")

tick 0: 'hello kafka streams'
       hello 1
       kafka 1
     streams 1
tick 1: 'all streams lead to kafka'
         all 1
     streams 2
        lead 1
          to 1
       kafka 2
tick 2: 'join kafka summit'
        join 1
       kafka 3
      summit 1


## Value-encoded records

The form above keeps the count inside the Z-set weight. To carry the count as an actual value, for serialization or for filtering and joining on it, use DeltaLiftedDeltaLiftedGroupBy, the incremental GROUP BY with aggregate. Group each word under itself and aggregate by summing weights. The per-tick output is the KTable changelog. It retracts the stale pair and asserts the fresh one, so integrating it rebuilds the absolute table.

In [4]:
from pydbsp.indexed_relational_operators import (
    DeltaLiftedDeltaLiftedGroupBy,
    LiftLiftIndex,
)
from pydbsp.indexed_zset import IndexedZSetAddition

g_rec = ZSetAddition[str]()
g_idx = IndexedZSetAddition[str, str](g_rec, lambda w: w)
g_out = ZSetAddition[tuple[str, int]]()

e = Evaluator[Time](
    circuit=Circuit[Time](),
    storage=DictStorage(),
    ctx=ComputeCtx(lattice=dbsp_time(2)),
    group=g_rec,
)
lines = Input[ZSet[str]](frontier=Antichain(dbsp_time(1))).connect(e.circuit, ())
words = Lift1[ZSet[str], ZSet[str]](f=to_words).connect(e.circuit, (lines,))
words_2d = LiftStreamIntroduction[ZSet[str]](group=g_rec).connect(e.circuit, (words,))
indexed = LiftLiftIndex[str, str](indexer=lambda w: w).connect(e.circuit, (words_2d,))
ktable = DeltaLiftedDeltaLiftedGroupBy[str, str, int](
    aggregate=lambda items: sum(w for _r, w in items),
    group=g_idx,
    out_group=g_out,
).connect(e.circuit, (indexed,))
ktable_cum = Integrate[ZSet[tuple[str, int]]](group=g_out).connect(e.circuit, (ktable,))

for t, line in enumerate(LINES):
    e.push(lines, ZSet({line: 1}))
    print(f"tick {t} changelog {dict(sorted(e.read(ktable, (t, 0)).inner.items()))}")

print()
print("final table", dict(sorted(e.read(ktable_cum, (len(LINES) - 1, 0)).inner.items())))

tick 0 changelog {('hello', 1): 1, ('kafka', 1): 1, ('streams', 1): 1}
tick 1 changelog {('all', 1): 1, ('kafka', 1): -1, ('kafka', 2): 1, ('lead', 1): 1, ('streams', 1): -1, ('streams', 2): 1, ('to', 1): 1}
tick 2 changelog {('join', 1): 1, ('kafka', 2): -1, ('kafka', 3): 1, ('summit', 1): 1}

final table {('all', 1): 1, ('hello', 1): 1, ('join', 1): 1, ('kafka', 3): 1, ('lead', 1): 1, ('streams', 2): 1, ('summit', 1): 1, ('to', 1): 1}


Watch kafka across the ticks. It moves from 1 to 2 to 3 as a clean stream of retraction and assertion.

## Retractions

The Kafka Streams example only ever adds, so counts only rise. DBSP Z-sets carry negative weights, so a correction is a push with weight -1, and the counts fall on their own. Suppose the second line was a mistake and we retract it.

In [5]:
g = ZSetAddition[str]()
e = Evaluator[Time](
    circuit=Circuit[Time](),
    storage=DictStorage(),
    ctx=ComputeCtx(lattice=dbsp_time(1)),
    group=g,
)
lines = Input[ZSet[str]](frontier=Antichain(dbsp_time(1))).connect(e.circuit, ())
words = Lift1[ZSet[str], ZSet[str]](f=to_words).connect(e.circuit, (lines,))
counts = Integrate[ZSet[str]](group=g).connect(e.circuit, (words,))

for line in LINES:
    e.push(lines, ZSet({line: 1}))
print("before", dict(sorted(e.latest(counts).inner.items())))

e.push(lines, ZSet({"all streams lead to kafka": -1}))
print("after ", dict(sorted(e.latest(counts).inner.items())))

before {'all': 1, 'hello': 1, 'join': 1, 'kafka': 3, 'lead': 1, 'streams': 2, 'summit': 1, 'to': 1}
after  {'hello': 1, 'join': 1, 'kafka': 2, 'streams': 1, 'summit': 1}


kafka falls from 3 to 2, streams from 2 to 1, and all, lead, and to disappear, while hello, join, and summit are untouched.

## Notes

Each push is one commit interval, so coarser batches give fewer and larger changelog updates, the same trade-off Kafka Streams exposes through commit.interval.ms. The Integrate retains the whole word table, like a KTable backed by RocksDB, and compact garbage-collects old timestamps while keeping the frontier snapshot that holds the live counts. The weight trick handles counts and sums only. Averages, top-k, and distinct counts need the value-encoded group-by. Windows key by window and word and let old buckets be retracted, which is a different construction than the unbounded Integrate used here.